In [14]:
PROJECT_REG = "mnist-mlp-reg"   #Proyecto para regularización
ENTITY = None

from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras import regularizers
from wandb.integration.keras import WandbMetricsLogger
import tensorflow as tf
import wandb

#Arquitectura fija del exp3: 784 -> 256(tanh) -> 128(tanh) -> 10(softmax)
def build_model_exp3_with_reg(cfg):
    reg = None
    if cfg["reg_type"] == "l1":
        reg = regularizers.l1(cfg["l1"])
    elif cfg["reg_type"] == "l2":
        reg = regularizers.l2(cfg["l2"])
    elif cfg["reg_type"] == "l1l2":
        reg = regularizers.l1_l2(l1=cfg["l1"], l2=cfg["l2"])

    model = Sequential([keras.Input(shape=(784,))])

    #Capa 1 (256 tanh) + regularizador
    model.add(Dense(256, activation="tanh",
                    kernel_initializer="glorot_uniform",
                    bias_initializer="zeros",
                    kernel_regularizer=reg))
    if cfg.get("dropout", 0.0) > 0:
        model.add(Dropout(cfg["dropout"]))

    #Capa 2 (128 tanh) + regularizador
    model.add(Dense(128, activation="tanh",
                    kernel_initializer="glorot_uniform",
                    bias_initializer="zeros",
                    kernel_regularizer=reg))
    if cfg.get("dropout", 0.0) > 0:
        model.add(Dropout(cfg["dropout"]))

    #Salida (softmax)
    model.add(Dense(10, activation="softmax"))

    #RMSprop (lr = 1e-3) + CCE
    opt = keras.optimizers.RMSprop(learning_rate=1e-3)
    model.compile(optimizer=opt, loss="categorical_crossentropy",
                  metrics=["categorical_accuracy"])
    return model

def run_exp3_reg(name, cfg, epochs=epochs, batch_size=batch_size):
    tf.keras.backend.clear_session()
    run = wandb.init(project=PROJECT_REG, entity=ENTITY, name=name,
                     config=cfg, reinit=True)

    model = build_model_exp3_with_reg(cfg)
    hist = model.fit(
        x_trainv, y_trainc,
        validation_data=(x_testv, y_testc),
        epochs=epochs, batch_size=batch_size, shuffle=True, verbose=1,
        callbacks=[WandbMetricsLogger(log_freq="epoch")]
    )

    preds = model.predict(x_testv, batch_size=1024, verbose=0).argmax(1)
    final_acc = float((preds == y_test).mean())
    wandb.log({"final_argmax_acc": final_acc})
    print(f"[{name}] test(argmax)={final_acc*100:.2f}%")
    print("Panel W&B:", wandb.run.url)
    wandb.finish()
    return hist, final_acc

#5 tipos de regularización sobre exp3
exp3_reg_variants = [
    ("exp3_reg_L1_1e-4",
     {"reg_type":"l1",   "l1":1e-4, "l2":0.0,   "dropout":0.0}),
    ("exp3_reg_L2_1e-4",
     {"reg_type":"l2",   "l1":0.0,  "l2":1e-4,  "dropout":0.0}),
    ("exp3_reg_L1L2_l1=1e-5_l2=1e-4",
     {"reg_type":"l1l2", "l1":1e-5, "l2":1e-4,  "dropout":0.0}),
    ("exp3_reg_Dropout_p=0.30",
     {"reg_type":"none", "l1":0.0,  "l2":0.0,   "dropout":0.30}),
    ("exp3_reg_Dropout_p=0.30_L1L2",
     {"reg_type":"l1l2", "l1":1e-5, "l2":1e-4,  "dropout":0.30}),
]

res_reg = []
for name, cfg in exp3_reg_variants:
    h, acc = run_exp3_reg(name, cfg, epochs=epochs, batch_size=batch_size)
    res_reg.append((name, acc))

print("\nResumen REG (test argmax):")
for name, acc in res_reg:
    print(f" - {name}: {acc*100:.2f}%")

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Epoch 1/30


I0000 00:00:1758133412.898684     128 service.cc:146] XLA service 0x7af0c8004f10 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1758133412.898710     128 service.cc:154]   StreamExecutor device (0): NVIDIA GeForce RTX 3050 Laptop GPU, Compute Capability 8.6
2025-09-17 18:23:32.937991: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-09-17 18:23:33.054609: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:531] Loaded cuDNN version 8906
2025-09-17 18:23:33.747801: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:393] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_125', 8 bytes spill stores, 8 bytes spill loads

2025-09-17 18:23:33.825271: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:393] ptxas warning : Registers are spilled to local memory i

 166/6000 ━━━━━━━━━━━━━━━━━━━━ 5s 920us/step - categorical_accuracy: 0.6738 - loss: 1.9364  

I0000 00:00:1758133415.201754     128 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


...

Epoch 1/30
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 6s 782us/step - categorical_accuracy: 0.8641 - loss: 0.4416 - val_categorical_accuracy: 0.9501 - val_loss: 0.1693
Epoch 2/30
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 4s 704us/step - categorical_accuracy: 0.9439 - loss: 0.1948 - val_categorical_accuracy: 0.9619 - val_loss: 0.1257
Epoch 3/30
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 4s 716us/step - categorical_accuracy: 0.9548 - loss: 0.1598 - val_categorical_accuracy: 0.9670 - val_loss: 0.1159
Epoch 4/30
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 4s 745us/step - categorical_accuracy: 0.9603 - loss: 0.1366 - val_categorical_accuracy: 0.9693 - val_loss: 0.1101
Epoch 5/30
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 5s 795us/step - categorical_accuracy: 0.9631 - loss: 0.1265 - val_categorical_accuracy: 0.9732 - val_loss: 0.1031
Epoch 6/30
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 5s 777us/step - categorical_accuracy: 0.9678 - loss: 0.1141 - val_categorical_accuracy: 0.9733 - val_loss: 0.0981
Epoch 7/30
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 4s 723us/step - catego

epoch/categorical_accuracy,▁▄▅▆▆▆▇▇▇▇▇▇▇▇▇█▇█████████████
epoch/epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▅▄▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_categorical_accuracy,▁▄▅▅▆▆▇▇▇▇▇▇▇▇▇▇▇█▇██▇█████▇▇█
epoch/val_loss,█▅▄▃▃▃▂▂▂▁▁▂▁▁▁▁▂▁▂▁▁▂▁▁▁▂▁▂▂▁
final_argmax_acc,▁
epoch/categorical_accuracy,0.98557
epoch/epoch,29
epoch/learning_rate,0.001
epoch/loss,0.0515


Epoch 1/30
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 7s 901us/step - categorical_accuracy: 0.8632 - loss: 0.5741 - val_categorical_accuracy: 0.9404 - val_loss: 0.2963
Epoch 2/30
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 5s 763us/step - categorical_accuracy: 0.9293 - loss: 0.3439 - val_categorical_accuracy: 0.9450 - val_loss: 0.2838
Epoch 3/30
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 4s 742us/step - categorical_accuracy: 0.9363 - loss: 0.3136 - val_categorical_accuracy: 0.9542 - val_loss: 0.2489
Epoch 4/30
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 5s 751us/step - categorical_accuracy: 0.9392 - loss: 0.3026 - val_categorical_accuracy: 0.9598 - val_loss: 0.2262
Epoch 5/30
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 4s 746us/step - categorical_accuracy: 0.9417 - loss: 0.2904 - val_categorical_accuracy: 0.9510 - val_loss: 0.2593
Epoch 6/30
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 5s 750us/step - categorical_accuracy: 0.9424 - loss: 0.2838 - val_categorical_accuracy: 0.9626 - val_loss: 0.2188
Epoch 7/30
6000/6000 ━━━━━━━━━━━━━━━━━━━━ 4s 734us/step - catego

epoch/categorical_accuracy,▁▆▇▇▇▇████████████████████████
epoch/epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▄▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_categorical_accuracy,▁▂▅▆▄▇▆▇▇▆▇▆█▆▇█▇▇▇▆▇▇▆██▇▇▆▆▆
epoch/val_loss,█▇▅▃▅▂▃▂▂▃▂▂▂▄▂▁▂▂▂▂▁▂▃▁▁▂▂▃▃▃
final_argmax_acc,▁
epoch/categorical_accuracy,0.9445
epoch/epoch,29
epoch/learning_rate,0.001
epoch/loss,0.27456



Resumen REG (test argmax):
 - exp3_reg_L1_1e-4: 96.91%
 - exp3_reg_L2_1e-4: 96.82%
 - exp3_reg_L1L2_l1=1e-5_l2=1e-4: 96.54%
 - exp3_reg_Dropout_p=0.30: 98.15%
 - exp3_reg_Dropout_p=0.30_L1L2: 95.92%
